<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name:Zhao Yibo
<br>
Github Username: Artmu-bo
<br>
USC ID: 5635641435

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import os, glob, re, warnings
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
warnings.filterwarnings('ignore')

Get the AReM Data Set

In [2]:
# Load all 88 instances
DATA_DIR = '../data/AReM'
COLS = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']

records = []
for activity in sorted(os.listdir(DATA_DIR)):
    folder = os.path.join(DATA_DIR, activity)
    if not os.path.isdir(folder):
        continue
    for path in glob.glob(os.path.join(folder, 'dataset*.csv')):
        num = int(re.search(r'dataset(\d+)\.csv', path).group(1))
        df = pd.read_csv(path, comment='#', header=None, sep=r'[,\s]+',
                         engine='python', usecols=range(7))
        df.columns = ['time'] + COLS
        records.append({'activity': activity, 'dataset': num, 'data': df})

records.sort(key=lambda r: (r['activity'], r['dataset']))
print('Total instances:', len(records))
print(pd.Series([r['activity'] for r in records]).value_counts().sort_index())

Total instances: 88
bending1     7
bending2     6
cycling     15
lying       15
sitting     15
standing    15
walking     15
Name: count, dtype: int64


**Data cleaning notes:** A few files required minor cleaning, which is handled directly in the parsing step:
- `bending2/dataset4.csv` is space-separated instead of comma-separated → handled by using `sep=r'[,\s]+'` (commas or whitespace).
- `cycling/dataset9.csv` and `cycling/dataset14.csv` have a trailing comma on the last line, creating an empty extra column → handled by `usecols=range(7)`.
- `sitting/dataset8.csv` contains 479 rows instead of 480 (one missing time step). Since all features are summary statistics, this does not affect the extraction, so the file is used as is.

### (b) Test and Train Data

In [3]:
# Split data
def is_test(r):
    if r['activity'] in ('bending1', 'bending2'):
        return r['dataset'] in (1, 2)
    return r['dataset'] in (1, 2, 3)

test_records = [r for r in records if is_test(r)]
train_records = [r for r in records if not is_test(r)]
print('Train:', len(train_records), '| Test:', len(test_records))

Train: 69 | Test: 19


### (c) Feature Extraction

#### i. Research

Commonly used time-domain features in time series classification include:
- **Central tendency:** mean, median, mode
- **Extremes:** minimum, maximum, range (max − min)
- **Dispersion:** standard deviation, variance, interquartile range (IQR), mean absolute deviation
- **Order statistics:** first quartile, third quartile, other percentiles
- **Distribution shape:** skewness, kurtosis
- **Energy / magnitude:** root mean square (RMS), signal energy, sum of absolute values
- **Temporal dynamics:** autocorrelation, zero-crossing / mean-crossing rate, number of peaks, slope or linear trend, sum of absolute differences between consecutive points
- **Complexity:** entropy (e.g., sample or approximate entropy)

#### ii. Extraction

In [4]:
# Extract min, max, mean, median, std, 1st and 3rd quartile from each of the 6 time series per instance
def extract_features(df):
    feats = {}
    for i, col in enumerate(COLS, start=1):
        s = df[col]
        feats[f'min{i}'] = s.min()
        feats[f'max{i}'] = s.max()
        feats[f'mean{i}'] = s.mean()
        feats[f'median{i}'] = s.median()
        feats[f'std{i}'] = s.std()
        feats[f'1st_quart{i}'] = s.quantile(0.25)
        feats[f'3rd_quart{i}'] = s.quantile(0.75)
    return feats

features = pd.DataFrame([extract_features(r['data']) for r in records])
features.insert(0, 'Instance', range(1, len(records) + 1))
features['activity'] = [r['activity'] for r in records]
features['set'] = ['test' if is_test(r) else 'train' for r in records]
features

,Instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6,activity,set
0,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,...,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000,bending1,test
1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,...,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000,bending1,test
2,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,...,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400,bending1,train
3,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,...,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000,bending1,train
4,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,...,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000,bending1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,...,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225,walking,train
84,85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,...,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650,walking,train
85,86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,...,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350,walking,train
86,87,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,...,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000,walking,train


#### iii. Standard Deviation

In [5]:
# Estimate the std of each feature and build a 90% bootstrap confidence interval (1000 resamples)
feat_cols = [c for c in features.columns if c not in ('Instance', 'activity', 'set')]
X = features[feat_cols].to_numpy()

np.random.seed(42)
B, n = 1000, len(X)
boot_stds = np.empty((B, X.shape[1]))
for b in range(B):
    idx = np.random.randint(0, n, n)
    boot_stds[b] = X[idx].std(axis=0, ddof=1)

pd.set_option('display.max_rows', 50)
std_ci = pd.DataFrame({
    'std': X.std(axis=0, ddof=1),
    'CI_lower (5%)': np.percentile(boot_stds, 5, axis=0),
    'CI_upper (95%)': np.percentile(boot_stds, 95, axis=0)
}, index=feat_cols).round(4)
std_ci

,std,CI_lower (5%),CI_upper (95%)
min1,9.5700,8.2674,10.6927
max1,4.3944,3.3341,5.2880
mean1,5.3357,4.6686,5.9027
median1,5.4401,4.7593,6.0179
std1,1.7722,1.5717,1.9459
1st_quart1,6.1536,5.5379,6.6505
3rd_quart1,5.1389,4.2755,5.8592
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6295,5.4063
mean2,1.5742,1.4024,1.7062


**Interpretation:** The table shows each feature's standard deviation with a 90% bootstrap confidence interval (1000 resamples). Features of the average signal series (1, 3, 5) generally vary more than those of the variance series (2, 4, 6), with `min1` (9.57) the largest. `min2` and `min4` have zero standard deviation (the minimum variance is 0 for every instance), so they carry no information for classification.

#### iv. Select Features

In [6]:
# Rank feature types by average ANOVA F-statistic across activities (train set only) to guide selection
train_feats = features[features['set'] == 'train']
F = pd.Series({c: f_oneway(*[g[c] for _, g in train_feats.groupby('activity')]).statistic
               for c in feat_cols})
F.groupby(F.index.str.replace(r'\d$', '', regex=True)).mean().sort_values(ascending=False)

mean         368.170192
3rd_quart    350.436027
median       317.745043
1st_quart    154.114553
std          109.167132
max           30.030112
min           14.638533
dtype: float64

I select **mean, 3rd quartile and standard deviation** as the three most important features.

- **Mean** has the highest average ANOVA F-statistic across activities on the training set, so it best separates the activity classes by signal level.
- **3rd quartile** ranks second. It captures the upper part of the distribution and is more robust to outliers than max (max and min have the lowest F-statistics because single spikes distort them).
- Median also scores high, but it is highly redundant with the mean, so I prefer **standard deviation**, which captures variability and distinguishes dynamic activities (walking, cycling) from static ones (sitting, standing, lying).

## 2. ISLR 3.7.4

### (a) Linear Train

The training RSS of the cubic regression is expected to be **lower than** (or at most equal to) that of the linear regression. The cubic model nests the linear model (β2 = β3 = 0), and least squares minimizes training RSS, so the extra flexibility lets the cubic fit also fit the noise(overfitting).

### (b) Linear Test

The test RSS of the **linear regression** is expected to be **lower**. Since the true relationship is linear, the cubic model's extra terms only fit noise in the training data (overfitting), which increases variance without reducing bias, leading to higher test RSS.

### (c) Not Linear Train

The training RSS of the **cubic regression** will be lower (or equal). Regardless of the true relationship, the more flexible nested model always achieves a training RSS no higher than the linear model.

### (d) Not Linear Testing

**There is not enough information to tell.** It depends on how far the true relationship is from linear. If it is substantially non-linear, the cubic model's lower bias should give a lower test RSS. If it is only slightly non-linear, the linear model may still have lower test RSS because the cubic model's added variance can outweigh its bias reduction.

## 3. ISLR 3.7.3 - Extra Practice 

The fitted model is:
$\hat{Y} = 50 + 20\,GPA + 0.07\,IQ + 35\,Level + 0.01\,(GPA \times IQ) - 10\,(GPA \times Level)$

**(a)** For fixed IQ and GPA, the salary difference between college (Level = 1) and high school (Level = 0) graduates is $35 - 10\,GPA$. This is negative when $GPA > 3.5$. So **iii is correct**: high school graduates earn more on average than college graduates provided that the GPA is high enough (above 3.5).

**(b)** $\hat{Y} = 50 + 20(4.0) + 0.07(110) + 35(1) + 0.01(4.0 \times 110) - 10(4.0 \times 1) = 50 + 80 + 7.7 + 35 + 4.4 - 40 = 137.1$, i.e. a predicted starting salary of **$137,100**.

**(c)** **False.** The size of a coefficient depends on the scale of the variables (GPA × IQ is on the order of hundreds, so a coefficient of 0.01 still changes salary by about 4.4 thousand dollars here). Evidence for an interaction effect should be judged by the coefficient's standard error, t-statistic and p-value, not by its magnitude.

## 3. ISLR 3.7.5 - Extra Practice 

Substituting $\hat{\beta}$ into $\hat{y}_i = x_i\hat{\beta}$:

$\hat{y}_i = x_i \frac{\sum_{i'=1}^{n} x_{i'} y_{i'}}{\sum_{k=1}^{n} x_k^2} = \sum_{i'=1}^{n} \frac{x_i x_{i'}}{\sum_{k=1}^{n} x_k^2}\, y_{i'}$

Therefore
$a_{i'} = \frac{x_i x_{i'}}{\sum_{k=1}^{n} x_k^2}$

(the summation index in the denominator is renamed to $k$ to avoid confusion with $i'$). Each fitted value is a linear combination of the response values.